In [1]:
!pip install ultralytics

In [9]:
import os
import shutil

# -------- PATHS --------
gt_file = "/mnt/DATA/TIH/Normal/IIT_DO_v7/gt/gt.txt"
image_dir = "/mnt/DATA/TIH/Normal/IIT_DO_v7/img1"

output_dir = "/mnt/DATA/TIH/Normal/IIT_DO_v7/new_dataset"
output_images = os.path.join(output_dir, "images")
output_labels = os.path.join(output_dir, "labels")

os.makedirs(output_images, exist_ok=True)
os.makedirs(output_labels, exist_ok=True)

# -------- IMAGE SIZE --------
IMG_WIDTH = 1920   # ⚠️ change if different
IMG_HEIGHT = 1080  # ⚠️ change if different

# -------- READ GT FILE --------
data = {}

with open(gt_file, "r") as f:
    for line in f:
        parts = line.strip().split(",")

        frame = int(parts[0])
        x, y, w, h = map(float, parts[2:6])

        if frame not in data:
            data[frame] = []

        data[frame].append((x, y, w, h))

# -------- PROCESS --------
for frame_id, boxes in data.items():

    # 🔥 IMPORTANT CHANGE HERE (zero padding)
    img_name = f"{frame_id:08d}.jpg"

    src_img_path = os.path.join(image_dir, img_name)

    if not os.path.exists(src_img_path):
        print(f"❌ Missing: {img_name}")
        continue

    # copy image
    dst_img_path = os.path.join(output_images, img_name)
    shutil.copy(src_img_path, dst_img_path)

    # create label
    label_name = f"{frame_id:08d}.txt"
    label_path = os.path.join(output_labels, label_name)

    with open(label_path, "w") as lf:
        for (x, y, w, h) in boxes:

            # Convert to YOLO
            x_center = (x + w / 2) / IMG_WIDTH
            y_center = (y + h / 2) / IMG_HEIGHT
            w_norm = w / IMG_WIDTH
            h_norm = h / IMG_HEIGHT

            class_id = 0

            lf.write(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n")

print("✅ Dataset created successfully!")

✅ Dataset created successfully!


In [11]:
import os
import shutil

# -------- PATHS --------
gt_file = "/mnt/DATA/TIH/Normal/IIT_DO_v1/gt/gt.txt"
image_dir = "/mnt/DATA/TIH/Normal/IIT_DO_v1/img1"

output_dir = "/mnt/DATA/TIH/Normal/IIT_DO_v1/new_dataset"
output_images = os.path.join(output_dir, "images")
output_labels = os.path.join(output_dir, "labels")

os.makedirs(output_images, exist_ok=True)
os.makedirs(output_labels, exist_ok=True)

# -------- IMAGE SIZE --------
IMG_WIDTH = 3840   # ⚠️ change if different
IMG_HEIGHT = 2160  # ⚠️ change if different

# -------- READ GT FILE --------
data = {}

with open(gt_file, "r") as f:
    for line in f:
        parts = line.strip().split(",")

        frame = int(parts[0])
        x, y, w, h = map(float, parts[2:6])

        if frame not in data:
            data[frame] = []

        data[frame].append((x, y, w, h))

# -------- PROCESS --------
for frame_id, boxes in data.items():

    # 🔥 IMPORTANT CHANGE HERE (zero padding)
    img_name = f"{frame_id:08d}.jpg"

    src_img_path = os.path.join(image_dir, img_name)

    if not os.path.exists(src_img_path):
        print(f"❌ Missing: {img_name}")
        continue

    # copy image
    dst_img_path = os.path.join(output_images, img_name)
    shutil.copy(src_img_path, dst_img_path)

    # create label
    label_name = f"{frame_id:08d}.txt"
    label_path = os.path.join(output_labels, label_name)

    with open(label_path, "w") as lf:
        for (x, y, w, h) in boxes:

            # Convert to YOLO
            x_center = (x + w / 2) / IMG_WIDTH
            y_center = (y + h / 2) / IMG_HEIGHT
            w_norm = w / IMG_WIDTH
            h_norm = h / IMG_HEIGHT

            class_id = 0

            lf.write(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n")

print("✅ Dataset created successfully!")

✅ Dataset created successfully!


In [13]:
import os
import shutil
from sklearn.model_selection import train_test_split

# =========================================================
# DATASET 1 (ONLY TRAIN)
# =========================================================
DATASET1_IMAGES = "/mnt/DATA/TIH/Normal/IIT_DO_v1/new_dataset/images"
DATASET1_LABELS = "/mnt/DATA/TIH/Normal/IIT_DO_v1/new_dataset/labels"

# =========================================================
# DATASET 2 (ONLY TRAIN)
# =========================================================
DATASET2_IMAGES = "/mnt/DATA/TIH/Normal/IIT_DO_v7/new_dataset/images"
DATASET2_LABELS = "/mnt/DATA/TIH/Normal/IIT_DO_v7/new_dataset/labels"

# =========================================================
# DATASET 3 (ONLY TRAIN)
# =========================================================
DATASET3_IMAGES = "/mnt/DATA/img_rgb"
DATASET3_LABELS = "/mnt/DATA/labels_rgb"

# =========================================================
# DATASET 4 (ALREADY SPLIT)
# =========================================================
DATASET4_PATH = "/mnt/DATA/our_data"  # contains train/val

# =========================================================
# FINAL OUTPUT
# =========================================================
FINAL_DATASET = "/mnt/DATA/final_dataset"

# Create folders
for split in ["train", "val"]:
    os.makedirs(os.path.join(FINAL_DATASET, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(FINAL_DATASET, split, "labels"), exist_ok=True)

# =========================================================
# SAFE COPY FUNCTION
# =========================================================
def copy_files(file_list, image_folder, label_folder, split_name, prefix):

    added, skipped = 0, 0

    for img_name in file_list:

        if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        label_name = os.path.splitext(img_name)[0] + ".txt"

        src_img = os.path.join(image_folder, img_name)
        src_label = os.path.join(label_folder, label_name)

        if not os.path.exists(src_label):
            skipped += 1
            continue

        new_img_name = prefix + "_" + img_name
        new_label_name = prefix + "_" + label_name

        dst_img = os.path.join(FINAL_DATASET, split_name, "images", new_img_name)
        dst_label = os.path.join(FINAL_DATASET, split_name, "labels", new_label_name)

        shutil.copy(src_img, dst_img)
        shutil.copy(src_label, dst_label)

        added += 1

    return added, skipped


# =========================================================
# FUNCTION TO SPLIT TRAIN DATA (80/20)
# =========================================================
def process_dataset(images_path, labels_path, prefix):

    print(f"\nProcessing {prefix}...")

    images = [
        f for f in os.listdir(images_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    train, val = train_test_split(images, test_size=0.2, random_state=42)

    a, s = copy_files(train, images_path, labels_path, "train", prefix)
    print(f"{prefix} Train: Added={a}, Skipped={s}")

    a, s = copy_files(val, images_path, labels_path, "val", prefix)
    print(f"{prefix} Val  : Added={a}, Skipped={s}")


# =========================================================
# PROCESS DATASET 1,2,3 (NEED SPLIT)
# =========================================================
process_dataset(DATASET1_IMAGES, DATASET1_LABELS, "d1")
process_dataset(DATASET2_IMAGES, DATASET2_LABELS, "d2")
process_dataset(DATASET3_IMAGES, DATASET3_LABELS, "d3")


# =========================================================
# PROCESS DATASET 4 (ALREADY SPLIT)
# =========================================================
print("\nProcessing Dataset 4 (already split)...")

for split in ["train", "val"]:

    img_folder = os.path.join(DATASET4_PATH, split, "images")
    lbl_folder = os.path.join(DATASET4_PATH, split, "labels")

    images = [
        f for f in os.listdir(img_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    a, s = copy_files(images, img_folder, lbl_folder, split, "d4")

    print(f"d4 {split}: Added={a}, Skipped={s}")


# =========================================================
# FINAL SUMMARY
# =========================================================
print("\n===================================")
print("FINAL DATASET CREATED")
print("===================================")

for split in ["train", "val"]:

    img_count = len(os.listdir(os.path.join(FINAL_DATASET, split, "images")))
    lbl_count = len(os.listdir(os.path.join(FINAL_DATASET, split, "labels")))

    print(f"\n{split.upper()}")
    print(f"Images : {img_count}")
    print(f"Labels : {lbl_count}")

print(f"\nSaved at: {FINAL_DATASET}")


Processing d1...
d1 Train: Added=2813, Skipped=0
d1 Val  : Added=704, Skipped=0

Processing d2...
d2 Train: Added=11592, Skipped=0
d2 Val  : Added=2898, Skipped=0

Processing d3...
d3 Train: Added=146, Skipped=1005
d3 Val  : Added=44, Skipped=244

Processing Dataset 4 (already split)...
d4 train: Added=684, Skipped=0
d4 val: Added=616, Skipped=0

FINAL DATASET CREATED

TRAIN
Images : 15235
Labels : 15235

VAL
Images : 4262
Labels : 4262

Saved at: /mnt/DATA/final_dataset


In [15]:
yaml_content = """
path: /mnt/DATA/final_dataset

train: train/images
val: val/images

nc: 1

names:
  0: drone
"""

# =========================================================
# SAVE YAML FILE
# =========================================================
yaml_path = "/mnt/DATA/final_dataset/data.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"data.yaml created at:\n{yaml_path}")

data.yaml created at:
/mnt/DATA/final_dataset/data.yaml


In [17]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO("yolo11n.pt")

# Train
model.train(
    data="/mnt/DATA/final_dataset/data.yaml",
    epochs=30,
    imgsz=1024,
    batch=16,
    scale=0.5,
    mosaic=1.0,
    mixup=0.2
)

Ultralytics 8.4.48 🚀 Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A4000, 15976MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/DATA/final_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-9, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x793a1c4ed2a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [2]:
from ultralytics import YOLO

model = YOLO("/mnt/DATA/runs/detect/train-9/weights/best.pt")

model.train(
    data="/mnt/DATA/final_dataset/data.yaml",
    epochs=5,          # 🔥 small
    lr0=1e-4,          # low learning rate
    resume=True
)

WARNING ⚠️ model '/mnt/DATA/runs/detect/train-9/weights/best.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.48 🚀 Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A4000, 15976MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/DATA/final_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x71b2dc734640>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [3]:
model = YOLO("/mnt/DATA/runs/detect/train-11/weights/best.pt")
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
results = model.val(save_json=True, plots=True)
model.predict(
source="/mnt/DATA/final_dataset/val/images",
save=True
)

Ultralytics 8.4.48 🚀 Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A4000, 15976MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5501.4±2025.6 MB/s, size: 219.6 KB)
val: Scanning /mnt/DATA/final_dataset/val/labels.cache... 4262 images, 88 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4262/4262 1.3Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 267/267 12.4it/s 21.5s0.1ss
                   all       4262       4221      0.967      0.892      0.918      0.823
Speed: 0.7ms preprocess, 2.9ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /mnt/DATA/runs/detect/val-7
mAP50: 0.9176495785068655
mAP50-95: 0.8228053687638403
Precision: 0.9673805787883273
Recall: 0.8922971158646197
Ultralytics 8.4.48 🚀 Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A4000, 15976MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, rea

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'drone'}
 obb: None
 orig_img: array([[[164, 165, 145],
         [163, 164, 144],
         [165, 166, 146],
         ...,
         [156, 162, 151],
         [154, 160, 149],
         [154, 160, 149]],
 
        [[165, 166, 146],
         [164, 165, 145],
         [166, 167, 147],
         ...,
         [154, 160, 149],
         [154, 160, 149],
         [155, 161, 150]],
 
        [[166, 167, 147],
         [164, 165, 145],
         [167, 168, 148],
         ...,
         [156, 162, 151],
         [155, 161, 150],
         [154, 160, 149]],
 
        ...,
 
        [[ 49,  66,  85],
         [ 51,  68,  87],
         [ 53,  70,  89],
         ...,
         [ 31,  51,  62],
         [ 30,  55,  65],
         [ 32,  59,  69]],
 
        [[ 49,  64,  83],
         [ 49,  64,  83],
         [ 48,  65,  84],
         ...,
         [ 31,  49,

In [5]:
from ultralytics import YOLO

# load trained model
model = YOLO("/mnt/DATA/runs/detect/train-11/weights/best.pt")

# run prediction on folder
results = model.predict(
    source="/mnt/DATA/split_rgb",   # your folder path
    conf=0.7,
    save=True,        # saves output images with detections
    save_txt=True     # saves label predictions (optional)
)

print("Testing completed on split_rgb images")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1439 /mnt/DATA/split_rgb/00000007.jpg: 1024x928 1 drone, 7.0ms
image 2/1439 /mnt/DATA/split_rgb/00000008.jpg: 1024x928 1 drone, 5.5ms
image 3/1439 /mnt/DATA/split_rgb/00000009.jpg: 1024x928 1 drone, 5.4ms
image 4/1439 /mnt/DATA/split_rgb/00000010.jpg: 1024x928 1 drone, 5.1ms
image 5/1439 /mnt/DATA/split_rgb/00000011.jpg: 1024x928 1 drone, 4.8ms
image 6/1439 /mnt/DATA/split_rgb/00000016.jpg: 1024x928 1 drone, 5.0ms
image 7/1439 /mnt/DATA/split_rgb

In [6]:

from ultralytics import YOLO

# load trained model
model = YOLO("/mnt/DATA/runs/detect/train-11/weights/best.pt")

# run prediction on folder
results = model.predict(
    source="/mnt/DATA/test_our/test",   # your folder path
    conf=0.25,
    save=True,        # saves output images with detections
    save_txt=True     # saves label predictions (optional)
)

print("Testing completed on split_rgb images")


image 1/614 /mnt/DATA/test_our/test/01002.png: 800x1024 (no detections), 5.3ms
image 2/614 /mnt/DATA/test_our/test/01003.png: 736x1024 1 drone, 7.9ms
image 3/614 /mnt/DATA/test_our/test/01006.png: 800x1024 1 drone, 5.0ms
image 4/614 /mnt/DATA/test_our/test/01016.png: 800x1024 1 drone, 5.3ms
image 5/614 /mnt/DATA/test_our/test/01033.png: 800x1024 1 drone, 5.5ms
image 6/614 /mnt/DATA/test_our/test/01038.png: 832x1024 1 drone, 6.1ms
image 7/614 /mnt/DATA/test_our/test/01041.png: 800x1024 1 drone, 6.5ms
image 8/614 /mnt/DATA/test_our/test/01042.png: 832x1024 1 drone, 6.1ms
image 9/614 /mnt/DATA/test_our/test/01054.png: 832x1024 (no detections), 5.2ms
image 10/614 /mnt/DATA/test_our/test/01062.png: 832x1024 2 drones, 5.2ms
image 11/614 /mnt/DATA/test_our/test/01063.png: 832x1024 1 drone, 4.9ms
image 12/614 /mnt/DATA/test_our/test/01064.png: 832x1024 1 drone, 5.4ms
image 13/614 /mnt/DATA/test_our/test/01085.png: 864x1024 (no detections), 6.5ms
image 14/614 /mnt/DATA/test_our/test/01101.png: